# CeView forecasting: Colab training and results

Run in a **Google Colab GPU runtime**. This notebook imports repository code to run the three baselines and one selected experiment per session, log results and best weights to Weights & Biases, and show training, validation, and the selected model's final test report.

Before running, place this project in your Google Drive at MyDrive/Ceview_Transformer. Include the ignored dataset, raw CSVs, and archived exporter listed in the frozen protocol, not just the Git checkout. Add your W&B key to the project .env; never paste it into a notebook cell.

The current dataset is provisional. Preflight will stop until actual collection verification, diagnostic approval, and data-owner permissions are recorded in a reviewed protocol revision. Do not set approval flags merely to bypass the check.

Set EXPERIMENT to "A" for the first session, "B" tomorrow, and "C" when ready. Each selection runs its three frozen seeds only. Use the same project, protocol, output folder, and compatible runtime across sessions.

Once setup and dataset review are complete, use **Runtime > Run all**. Drive may request access. Completed runs are reused; an interrupted training run starts a new same-seed attempt. A completed test report is displayed again without reevaluating the test set.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Experiment selection and locations
Change these paths if your Drive folders use different names. Keep OUTPUT_ROOT unchanged across reruns so the test ledger persists.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

EXPERIMENT = "A"  # Choose A today, B tomorrow, or C in a later session.

PROJECT_ROOT = Path("/content/drive/MyDrive/Ceview_Transformer")
OUTPUT_ROOT = Path("/content/drive/MyDrive/CeviewTraining")
PROTOCOL_PATH = PROJECT_ROOT / "experiments/forecasting-v3-approved.json"

if not (PROJECT_ROOT / "training/requirements.txt").is_file():
    raise FileNotFoundError("Copy the project into PROJECT_ROOT before continuing.")
if not PROTOCOL_PATH.is_file():
    raise FileNotFoundError("The selected frozen protocol is missing.")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

## Dependencies
Install before importing the model packages. If preflight detects an already-loaded incompatible package, restart the runtime and run all cells again.

In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(PROJECT_ROOT / "training/requirements.txt")
])

## Configuration and preflight
Only non-secret configuration is displayed. .env is loaded by the logging helper and is never included in W&B artifacts.

In [ ]:
import torch
import pandas as pd
from IPython.display import display
from model.experiment import load_protocol
from training.tracking import configure_wandb
from training.workflow import preflight, run_suite, display_results

protocol = load_protocol(PROTOCOL_PATH)
display(pd.DataFrame([{
    "protocol": protocol["protocol_version"],
    "dataset": protocol["dataset"]["version"],
    "status": protocol["dataset"]["status"],
    "diagnostic_approval": protocol["dataset"]["diagnostic_approval"],
    "training_examples": protocol["dataset"]["counts"]["train"],
    "validation_examples": protocol["dataset"]["counts"]["validation"],
    "test_examples": protocol["dataset"]["counts"]["test"],
    "GPU_available": torch.cuda.is_available(),
}]))
device = torch.device("cuda")
preflight(PROJECT_ROOT, PROTOCOL_PATH, device)
configure_wandb(PROJECT_ROOT / ".env")
print("Preflight passed; credentials loaded without displaying the key.")

## Run the selected experiment

Epoch MAE values appear below and stream to W&B. The runner saves each best validation checkpoint to Drive and uploads it as a W&B artifact on run completion. It saves the selected experiment's training and validation results, then stops if other experiments are incomplete. Once all nine runs across A/B/C exist, it records validation-only selection before opening the test partition. If no architecture beats the strongest baseline, it reports that result and leaves test data sealed.

In [ ]:
suite = run_suite(PROJECT_ROOT, PROTOCOL_PATH, OUTPUT_ROOT, device=device, experiment=EXPERIMENT)

## Results
Show all completed training curves, per-seed scores, detailed validation results, baselines, and W&B links, including partial A-only or A+B progress. Architecture comparison and final test scores appear only after all three experiments are complete.

In [ ]:
display_results(suite)

In [ ]:
from pathlib import Path
from datetime import datetime
from IPython.utils.capture import capture_output
from google.colab import files
import matplotlib.pyplot as plt
import base64
import html
import shutil

output_dir = Path("/content") / datetime.now().strftime(
    "training_results_%Y%m%d_%H%M%S_%f"
)
output_dir.mkdir(parents=True)

# Capture the existing results and graphs.
with capture_output() as captured:
    display_results(suite)
    plt.show()

captured.show()

report = [
    "<!DOCTYPE html><html><head><meta charset='utf-8'>",
    "<title>Training results</title>",
    "<style>body{font-family:Arial;padding:24px}"
    "table{border-collapse:collapse;margin:20px 0}"
    "th,td{border:1px solid #ccc;padding:8px}"
    "img{max-width:100%;height:auto}pre{white-space:pre-wrap}</style>",
    "</head><body><h1>Training results</h1>",
]

for index, output in enumerate(captured.outputs, start=1):
    data = output.data

    if "image/png" in data:
        encoded = data["image/png"]
        if isinstance(encoded, bytes):
            encoded = encoded.decode("ascii")
        (output_dir / f"graph_{index:02d}.png").write_bytes(
            base64.b64decode(encoded)
        )
        report.append(f'<img src="data:image/png;base64,{encoded}">')
    elif "image/svg+xml" in data:
        svg = data["image/svg+xml"]
        (output_dir / f"graph_{index:02d}.svg").write_text(
            svg, encoding="utf-8"
        )
        report.append(svg)
    elif "text/html" in data:
        report.append(data["text/html"])
    elif "text/plain" in data:
        report.append(f"<pre>{html.escape(data['text/plain'])}</pre>")

for text in (captured.stdout, captured.stderr):
    if text:
        report.append(f"<pre>{html.escape(text)}</pre>")

report.append("</body></html>")
(output_dir / "report.html").write_text(
    "\n".join(report), encoding="utf-8"
)

zip_path = shutil.make_archive(str(output_dir), "zip", output_dir)
print(f"Saved to: {output_dir}")
files.download(zip_path)

## Download the complete model after training

This cell runs after the results display. It exports the best completed architecture using mean validation MAE across all three seeds and the protocol's fixed final seed. The full model uses that seed's best validation checkpoint. Until final selection, the bundle is labeled provisional; a model that fails the baseline gate is labeled no_qualifying_champion.

A new best model triggers one ZIP download, with a persistent copy in Drive and a copy in `/content` in Colab's Files panel. An unchanged model stays available for manual download without repeating the automatic download. No export runs during individual epochs. Existing training checkpoints still save to Drive for recovery.

The ZIP contains `model.pt` (the complete model with trained parameters), the required model classes, a loader, protocol, preprocessing instructions, and saved scores. Keep the extracted files together. This follows [PyTorch's full-model serialization](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html). Test scores are included only when the existing final evaluation is available; this cell never evaluates test data.


In [ ]:
from google.colab import files
from model.architectures import Forecaster
from model.experiment import file_digest
from training.workflow import read_json, write_json
import json
import shutil
import tempfile

# Compare completed architectures using validation only and the frozen final seed.
completed = []
for architecture in ("A", "B", "C"):
    paths = [suite / f"{architecture}-{seed}" / "result.json"
             for seed in protocol["training"]["seeds"]]
    if all(path.is_file() for path in paths):
        rows = [read_json(path) for path in paths]
        completed.append({
            "architecture": architecture,
            "mae": sum(r["validation"]["overall"]["mae"] for r in rows) / len(rows),
            "smape": sum(r["validation"]["overall"]["smape_percent"] for r in rows) / len(rows),
            "parameters": rows[0]["parameter_count"],
        })
if not completed:
    raise RuntimeError("Finish training an experiment before exporting a model.")
if not all(torch.isfinite(torch.tensor([r["mae"], r["smape"]])).all() for r in completed):
    raise ValueError("Cannot export a best model with nonfinite validation scores.")
winner = min(completed, key=lambda r: (r["mae"], r["smape"], r["parameters"], r["architecture"]))
selection_path = suite / "selection.json"
selection = read_json(selection_path) if selection_path.exists() else None
status = "provisional"
if selection is not None:
    if selection["architecture"]:
        if selection["architecture"] != winner["architecture"]:
            raise ValueError("Saved selection disagrees with validation results.")
        status = "selected"
    else:
        status = "no_qualifying_champion"

architecture = winner["architecture"]
seed = protocol["selection"]["final_seed"]
run_dir = suite / f"{architecture}-{seed}"
result = read_json(run_dir / "result.json")
checkpoint = (run_dir / result["checkpoint"]).resolve()
if not checkpoint.is_relative_to(run_dir.resolve()) or file_digest(checkpoint) != result["checkpoint_sha256"]:
    raise ValueError("Best checkpoint is missing or changed.")
identity = {"checkpoint_sha256": result["checkpoint_sha256"],
            "protocol_sha256": protocol["protocol_sha256"], "status": status}
export_root = suite / "model_exports"
export_root.mkdir(exist_ok=True)
name = f"ceview_{architecture}_seed{seed}_{status}_{result['checkpoint_sha256'][:12]}"
archive = export_root / f"{name}.zip"
marker = export_root / f"{name}.json"
record = read_json(marker) if marker.exists() else {}
new_export = (record.get("identity") != identity or not archive.is_file()
              or record.get("archive_sha256") != file_digest(archive))
if new_export:
    saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
    if saved["protocol_sha256"] != protocol["protocol_sha256"]:
        raise ValueError("Checkpoint belongs to another protocol.")
    full_model = Forecaster(architecture, PROTOCOL_PATH).cpu().eval()
    full_model.load_state_dict(saved["state_dict"])
    with tempfile.TemporaryDirectory() as temporary:
        bundle = Path(temporary) / name
        bundle.mkdir()
        torch.save(full_model, bundle / "model.pt")
        # Verify serialization with synthetic inputs; do not reopen any data partition.
        restored = torch.load(bundle / "model.pt", map_location="cpu", weights_only=False).eval()
        sample = (torch.zeros(2, protocol["dataset"]["lookback"], len(protocol["dataset"]["features"])),
                  torch.zeros(2, dtype=torch.int64), torch.zeros(2, dtype=torch.int64))
        with torch.no_grad():
            torch.testing.assert_close(restored(*sample), full_model(*sample))
        (bundle / "model").mkdir()
        for filename in ("__init__.py", "architectures.py", "components.py", "experiment.py"):
            shutil.copy2(PROJECT_ROOT / "model" / filename, bundle / "model" / filename)
        shutil.copy2(PROTOCOL_PATH, bundle / "protocol.json")
        shutil.copy2(PROJECT_ROOT / "docs/preprocessing.md", bundle / "preprocessing.md")
        write_json(bundle / "metrics.json", {"status": status, "architecture_comparison": completed,
                                           "exported_run": result})
        if selection is not None:
            shutil.copy2(selection_path, bundle / "selection.json")
        if status == "selected" and (suite / "test.json").is_file():
            shutil.copy2(suite / "test.json", bundle / "test.json")
        (bundle / "requirements.txt").write_text(f"torch=={torch.__version__.split('+')[0]}\n", encoding="utf-8")
        (bundle / "load_model.py").write_text(
            'from pathlib import Path\nimport torch\n\n'
            'def load_model():\n'
            '    # Load only this trusted, locally generated model bundle.\n'
            '    path = Path(__file__).resolve().parent / "model.pt"\n'
            '    return torch.load(path, map_location="cpu", weights_only=False).eval()\n',
            encoding="utf-8")
        (bundle / "README.md").write_text(
            f"# CeView complete model\n\nStatus: {status}. Architecture: {architecture}. Seed: {seed}.\n\n"
            "Extract the entire ZIP and keep its files together. Install requirements.txt, then run Python "
            "from the extracted folder:\n\n```python\nfrom load_model import load_model\nmodel = load_model()\n```\n\n"
            "model.pt contains the full PyTorch model and its trained parameters. The model folder supplies "
            "its class definitions; the original project is not required. Only load trusted bundles.\n\n"
            "Call model(history, market_id, category_id) with preprocessed float32 history [batch, 52, 3] "
            "and int64 identity vectors [batch]. Use the feature order and vocabulary in protocol.json "
            "and the rules in preprocessing.md. Outputs are 12 raw scaled forecasts; multiply by 100 "
            "and clamp to [0, 100] for display.\n\n"
            "The architecture uses mean validation MAE across its three seeds, with the protocol's tie-breaks "
            "and fixed final seed. Its weights come from that seed's best validation epoch. Provisional "
            "and no_qualifying_champion exports are for inspection, not approved champions. Test scores "
            "appear only for the final selected model and never determine the export winner.\n\n"
            "Serialization reference: https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html\n",
            encoding="utf-8")
        temporary_archive = Path(shutil.make_archive(str(Path(temporary) / "download"), "zip", bundle))
        pending = archive.with_suffix(".tmp")
        shutil.copy2(temporary_archive, pending)
        pending.replace(archive)
    write_json(marker, {"identity": identity, "archive_sha256": file_digest(archive)})

local_archive = Path("/content") / archive.name
shutil.copy2(archive, local_archive)
print(f"Model status: {status}. Mean validation MAE: {winner['mae']:.4f}")
print(f"Colab Files: {local_archive}")
print(f"Persistent Drive copy: {archive}")
if new_export:
    try:
        files.download(str(local_archive))
    except Exception as error:
        print(f"Automatic download unavailable ({error}). Download the ZIP from Colab's Files panel.")
else:
    print("Best model is unchanged. The ZIP is available in Colab's Files panel for download.")


## Saved outputs

Drive contains the suite record, baseline metrics, per-run epoch CSVs and best weights, selection record, and final test JSON. W&B contains corresponding runs, tables, and explicitly selected model/evaluation artifacts. Dataset files, .env, and source folders are not uploaded as artifacts.

The persistent test-evaluations folder guards against automatic repeated test evaluation. If a runtime fails after claiming the test but before saving the report, the runner stops for a ledger review instead of silently reevaluating. Do not delete the ledger to compare another candidate against test results.